In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

#  Install libraries
!pip install ultralytics -q
!pip install rasterio -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.8 MB/s eta 0:00:00


**COMPLETE ENHANCED Feature Extraction for Crop Yield Optimization With Canopy Column and Analysis**

In [ ]:

"""
Install: !pip install tensorflow keras rasterio numpy pandas opencv-python pillow scikit-learn
"""

import os, numpy as np, pandas as pd, rasterio
from pathlib import Path
import cv2, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB4, ResNet50, VGG16
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as res_preprocess
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess

print(f"TensorFlow: {tf.__version__} | GPU: {len(tf.config.list_physical_devices('GPU'))}")

# ==================== DRIVE MOUNT ====================
def mount_google_drive():
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print(" Drive mounted!")
        return True
    except:
        print("  Not in Colab environment")
        return False

def find_crops_folder():
    """Search for Crops folder in Google Drive"""
    print("🔍 Searching for Crops folder in Google Drive...")

    search_paths = [
        '/content/drive/MyDrive/Crops',
        '/content/drive/MyDrive/Final Year Project/Crops',
        '/content/drive/MyDrive/FYP/Crops',
        '/content/drive/Shareddrives/*/Crops',
    ]

    for path in search_paths:
        if '*' in path:
            from glob import glob
            matches = glob(path)
            if matches:
                print(f" Found: {matches[0]}")
                return matches[0]
        elif os.path.exists(path) and os.path.isdir(path):
            print(f" Found: {path}")
            return path

    # Deep search in MyDrive
    print(" Deep searching in MyDrive...")
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if 'Crops' in dirs:
            full_path = os.path.join(root, 'Crops')
            print(f" Found: {full_path}")
            return full_path

    print(" Crops folder not found!")
    return None

# ==================== CNN MODEL SELECTOR ====================
class CNNFeatureExtractor:
    """Multi-model feature extractor with model comparison"""

    MODELS = {
        'efficientnet-b4': {
            'model': EfficientNetB4,
            'preprocess': eff_preprocess,
            'size': (380, 380),
            'features': 1792
        },
        'resnet50': {
            'model': ResNet50,
            'preprocess': res_preprocess,
            'size': (224, 224),
            'features': 2048
        },
        'vgg16': {
            'model': VGG16,
            'preprocess': vgg_preprocess,
            'size': (224, 224),
            'features': 4096
        }
    }

    def __init__(self, model_name='efficientnet-b4'):
        self.model_name = model_name
        config = self.MODELS[model_name]

        print(f"\n Loading {model_name.upper()}...")
        self.model = config['model'](weights='imagenet', include_top=False, pooling='avg')
        self.preprocess_fn = config['preprocess']
        self.input_size = config['size']
        self.feature_dim = config['features']
        print(f" Loaded! Features: {self.feature_dim}")

    def extract_features(self, image):
        img = cv2.resize(image, self.input_size)
        img = np.expand_dims(img, axis=0)
        img = self.preprocess_fn(img)
        features = self.model.predict(img, verbose=0)
        return features.flatten()

# ==================== FILE LOADING ====================
def load_single_band_tiff(file_path):
    """Load TIFF file with proper error handling"""
    try:
        with rasterio.open(file_path) as src:
            data = src.read(1).astype(np.float32)
            # Replace NoData values with NaN
            if src.nodata is not None:
                data[data == src.nodata] = np.nan
            return data
    except Exception as e:
        print(f"        Error loading {file_path}: {e}")
        return None

def find_file(phase_folder, keywords):
    """Find file by keywords (case-insensitive)"""
    for ext in ['*.tif', '*.tiff', '*.TIF', '*.TIFF']:
        for file in phase_folder.glob(ext):
            name_upper = file.name.upper()
            if any(kw.upper() in name_upper for kw in keywords):
                return file
    return None

# ==================== RGB COMPOSITE CREATION ====================
def create_rgb_from_bands(phase_folder, debug=False):
    """Create RGB composite with flexible band matching"""

    band_search = {
        'B02': ['B02', 'B2', 'BLUE'],
        'B03': ['B03', 'B3', 'GREEN'],
        'B04': ['B04', 'B4', 'RED'],
        'B08': ['B08', 'B8', 'NIR'],
        'B11': ['B11', 'SWIR1'],
        'B12': ['B12', 'SWIR2']
    }

    loaded = {}
    for band_name, keywords in band_search.items():
        file = find_file(phase_folder, keywords)
        if file:
            try:
                data = load_single_band_tiff(str(file))
                if data is not None:
                    loaded[band_name] = data
                    if debug:
                        print(f"      ✓ {band_name}: {file.name}")
            except Exception as e:
                if debug:
                    print(f"      ✗ {band_name}: Error - {e}")

    if not loaded:
        return None, {}

    # Create composite
    if all(k in loaded for k in ['B04', 'B03', 'B02']):
        r, g, b = loaded['B04'], loaded['B03'], loaded['B02']
        mode = "True Color"
    elif all(k in loaded for k in ['B08', 'B04', 'B03']):
        r, g, b = loaded['B08'], loaded['B04'], loaded['B03']
        mode = "False Color"
    else:
        vals = list(loaded.values())
        r = vals[0] if len(vals) > 0 else np.zeros((100, 100))
        g = vals[1] if len(vals) > 1 else r
        b = vals[2] if len(vals) > 2 else g
        mode = "Partial"

    # Stack and normalize with proper NaN handling
    image = np.stack([r, g, b], axis=-1)
    image = np.nan_to_num(image, nan=0.0)

    for i in range(3):
        band = image[:, :, i]
        valid_pixels = band[band > 0]
        if len(valid_pixels) > 0:
            p2, p98 = np.percentile(valid_pixels, [2, 98])
            if p98 > p2:
                image[:, :, i] = np.clip((band - p2) / (p98 - p2 + 1e-8) * 255, 0, 255)
            else:
                image[:, :, i] = np.clip(band / (np.max(band) + 1e-8) * 255, 0, 255)

    if debug:
        print(f"       RGB created: {mode}")

    return image.astype(np.uint8), loaded

# ==================== CANOPY FEATURE CALCULATION ====================
def calculate_canopy_features(bands_dict, raw_indices):
    """Calculate canopy-related features"""
    canopy_features = {}

    try:
        # Canopy Health Index (based on NDVI and EVI)
        if 'NDVI' in raw_indices and 'EVI' in raw_indices:
            ndvi_data = raw_indices['NDVI']
            evi_data = raw_indices['EVI']

            valid_mask = (~np.isnan(ndvi_data)) & (~np.isnan(evi_data))
            if np.sum(valid_mask) > 100:
                ndvi_valid = ndvi_data[valid_mask]
                evi_valid = evi_data[valid_mask]

                ndvi_mean = np.mean(ndvi_valid)
                evi_mean = np.mean(evi_valid)

                # Canopy Health Index (combination of NDVI and EVI)
                canopy_features['canopy_health'] = (ndvi_mean + evi_mean) / 2.0
            else:
                canopy_features['canopy_health'] = np.nan
        else:
            canopy_features['canopy_health'] = np.nan

        # Canopy Density (from NIR reflectance)
        if 'B08' in bands_dict:
            nir_data = bands_dict['B08']
            valid_data = nir_data[~np.isnan(nir_data)]
            if len(valid_data) > 0:
                nir_mean = np.mean(valid_data)
                # Normalize NIR to 0-1 range for canopy density
                canopy_features['canopy_density'] = np.clip(nir_mean / 10000.0, 0, 1)
            else:
                canopy_features['canopy_density'] = np.nan
        else:
            canopy_features['canopy_density'] = np.nan

        # Canopy Water Content (from NDWI and Moisture)
        if 'NDWI' in raw_indices:
            ndwi_data = raw_indices['NDWI']
            valid_data = ndwi_data[~np.isnan(ndwi_data)]
            if len(valid_data) > 0:
                ndwi_mean = np.mean(valid_data)
                canopy_features['canopy_water_content'] = np.clip((ndwi_mean + 1) / 2.0, 0, 1)
            else:
                canopy_features['canopy_water_content'] = np.nan
        else:
            canopy_features['canopy_water_content'] = np.nan

        # Canopy Stress Index (combination of multiple stress indicators)
        stress_components = []

        if 'NDVI' in raw_indices:
            ndvi_data = raw_indices['NDVI']
            valid_data = ndvi_data[~np.isnan(ndvi_data)]
            if len(valid_data) > 0:
                ndvi_mean = np.mean(valid_data)
                # Lower NDVI indicates higher stress
                stress_components.append(1.0 - np.clip((ndvi_mean + 1) / 2.0, 0, 1))

        if 'NDWI' in raw_indices:
            ndwi_data = raw_indices['NDWI']
            valid_data = ndwi_data[~np.isnan(ndwi_data)]
            if len(valid_data) > 0:
                ndwi_mean = np.mean(valid_data)
                # Lower NDWI indicates higher water stress
                stress_components.append(1.0 - np.clip((ndwi_mean + 1) / 2.0, 0, 1))

        if stress_components:
            canopy_features['canopy_stress'] = np.mean(stress_components)
        else:
            canopy_features['canopy_stress'] = np.nan

        # Overall Canopy Index (composite score)
        positive_indicators = []
        if 'canopy_health' in canopy_features and not np.isnan(canopy_features['canopy_health']):
            positive_indicators.append(np.clip((canopy_features['canopy_health'] + 1) / 2.0, 0, 1))
        if 'canopy_density' in canopy_features and not np.isnan(canopy_features['canopy_density']):
            positive_indicators.append(canopy_features['canopy_density'])
        if 'canopy_water_content' in canopy_features and not np.isnan(canopy_features['canopy_water_content']):
            positive_indicators.append(canopy_features['canopy_water_content'])

        if positive_indicators:
            canopy_features['canopy_index'] = np.mean(positive_indicators)
        else:
            canopy_features['canopy_index'] = np.nan

    except Exception as e:
        print(f"        Error in canopy features: {e}")
        canopy_features = {
            'canopy_health': np.nan,
            'canopy_density': np.nan,
            'canopy_water_content': np.nan,
            'canopy_stress': np.nan,
            'canopy_index': np.nan
        }

    return canopy_features

# ==================== SPECTRAL INDICES EXTRACTION ====================
def extract_spectral_indices(phase_folder, debug=False):
    """Extract vegetation and water stress indices with robust NaN handling"""

    index_search = {
        'NDVI': ['NDVI'],
        'EVI': ['EVI'],
        'SAVI': ['SAVI'],
        'NDWI': ['NDWI', 'WATER'],
        'LSWI': ['LSWI', 'LAND_SURFACE_WATER', 'LSW'],
        'Moisture': ['MOISTURE', 'MSI', 'STRESS'],
        'SWIR': ['SWIR', 'B11', 'B12']
    }

    stats = {}
    raw_data = {}

    for idx_name, keywords in index_search.items():
        file = find_file(phase_folder, keywords)
        if file:
            try:
                data = load_single_band_tiff(str(file))
                if data is None:
                    continue

                # Normalize if data is in uint16 range
                if np.nanmax(data) > 10 and np.nanmax(data) <= 65535:
                    data_normalized = (data / 65535.0) * 2.0 - 1.0
                    raw_data[idx_name] = data_normalized
                    data = data_normalized
                    if debug:
                        print(f"      ✓ {idx_name}: {file.name} (normalized from uint16)")
                else:
                    raw_data[idx_name] = data
                    if debug:
                        print(f"      ✓ {idx_name}: {file.name}")

                # Remove zeros and NaN values for statistics
                valid_mask = (data != 0) & (~np.isnan(data))
                valid_data = data[valid_mask]

                if len(valid_data) > 0:
                    stats[f'{idx_name}_mean'] = np.mean(valid_data)
                    stats[f'{idx_name}_std'] = np.std(valid_data)
                    stats[f'{idx_name}_median'] = np.median(valid_data)
                    stats[f'{idx_name}_min'] = np.min(valid_data)
                    stats[f'{idx_name}_max'] = np.max(valid_data)

                    # Coefficient of variation
                    mean_val = stats[f'{idx_name}_mean']
                    if mean_val != 0:
                        stats[f'{idx_name}_cv'] = stats[f'{idx_name}_std'] / abs(mean_val)
                    else:
                        stats[f'{idx_name}_cv'] = np.nan

                    stats[f'{idx_name}_range'] = stats[f'{idx_name}_max'] - stats[f'{idx_name}_min']

                else:
                    # No valid data
                    stats.update({f'{idx_name}_{stat}': np.nan for stat in
                                ['mean', 'std', 'median', 'min', 'max', 'cv', 'range']})

            except Exception as e:
                if debug:
                    print(f"      ✗ {idx_name}: Error - {e}")
                stats.update({f'{idx_name}_{stat}': np.nan for stat in
                            ['mean', 'std', 'median', 'min', 'max', 'cv', 'range']})
        else:
            if debug:
                print(f"        {idx_name}: File not found")

    return stats, raw_data

# ==================== CUSTOM STRESS INDICES CALCULATION ====================
def calculate_custom_stress_indices(bands_dict, raw_indices):
    """Calculate custom stress indices as per desired output format"""
    stress_indices = {}

    try:
        # CWSI Proxy (Crop Water Stress Index)
        if 'NDVI' in raw_indices and 'NDWI' in raw_indices:
            ndvi_data = raw_indices['NDVI']
            ndwi_data = raw_indices['NDWI']

            valid_mask = (~np.isnan(ndvi_data)) & (~np.isnan(ndwi_data))

            if np.sum(valid_mask) > 100:
                ndvi_valid = ndvi_data[valid_mask]
                ndwi_valid = ndwi_data[valid_mask]

                ndvi_mean = np.mean(ndvi_valid)
                ndwi_mean = np.mean(ndwi_valid)

                # Normalize to 0-1 range
                ndvi_norm = np.clip((ndvi_mean + 1) / 2.0, 0, 1)
                ndwi_norm = np.clip((ndwi_mean + 1) / 2.0, 0, 1)

                stress_indices['cwsi_proxy'] = np.clip(1.0 - (ndvi_norm * ndwi_norm), 0, 1)
            else:
                stress_indices['cwsi_proxy'] = np.nan
        else:
            stress_indices['cwsi_proxy'] = np.nan

        # CWSI from SWIR/NIR ratio
        if 'B08' in bands_dict and 'B11' in bands_dict:
            nir_data = bands_dict['B08']
            swir_data = bands_dict['B11']

            valid_mask = (nir_data != 0) & (swir_data != 0) & (~np.isnan(nir_data)) & (~np.isnan(swir_data))

            if np.sum(valid_mask) > 100:
                nir_valid = nir_data[valid_mask]
                swir_valid = swir_data[valid_mask]

                nir_mean = np.mean(nir_valid)
                swir_mean = np.mean(swir_valid)

                if nir_mean > 0.001:
                    stress_indices['cwsi_swir_nir'] = swir_mean / nir_mean
                else:
                    stress_indices['cwsi_swir_nir'] = np.nan
            else:
                stress_indices['cwsi_swir_nir'] = np.nan
        else:
            stress_indices['cwsi_swir_nir'] = np.nan

        # LSWI deficit
        if 'LSWI' in raw_indices and 'NDVI' in raw_indices:
            lswi_data = raw_indices['LSWI']
            ndvi_data = raw_indices['NDVI']

            valid_mask = (~np.isnan(lswi_data)) & (~np.isnan(ndvi_data))

            if np.sum(valid_mask) > 100:
                lswi_valid = lswi_data[valid_mask]
                ndvi_valid = ndvi_data[valid_mask]

                lswi_mean = np.mean(lswi_valid)
                ndvi_mean = np.mean(ndvi_valid)

                stress_indices['lswi_deficit'] = ndvi_mean - lswi_mean

                if ndvi_mean != 0:
                    stress_indices['lswi_ndvi_ratio'] = lswi_mean / ndvi_mean
                else:
                    stress_indices['lswi_ndvi_ratio'] = np.nan
            else:
                stress_indices['lswi_deficit'] = np.nan
                stress_indices['lswi_ndvi_ratio'] = np.nan
        else:
            stress_indices['lswi_deficit'] = np.nan
            stress_indices['lswi_ndvi_ratio'] = np.nan

        # NDWI Stress (1 - NDWI normalized)
        if 'NDWI' in raw_indices:
            ndwi_data = raw_indices['NDWI']
            valid_data = ndwi_data[~np.isnan(ndwi_data)]
            if len(valid_data) > 0:
                ndwi_mean = np.mean(valid_data)
                ndwi_norm = np.clip((ndwi_mean + 1) / 2.0, 0, 1)
                stress_indices['ndwi_stress'] = 1.0 - ndwi_norm
            else:
                stress_indices['ndwi_stress'] = np.nan
        else:
            stress_indices['ndwi_stress'] = np.nan

        # Moisture Stress (1 - Moisture normalized)
        if 'Moisture' in raw_indices:
            moisture_data = raw_indices['Moisture']
            valid_data = moisture_data[~np.isnan(moisture_data)]
            if len(valid_data) > 0:
                moisture_mean = np.mean(valid_data)
                moisture_norm = np.clip((moisture_mean + 1) / 2.0, 0, 1)
                stress_indices['moisture_stress'] = 1.0 - moisture_norm
            else:
                stress_indices['moisture_stress'] = np.nan
        else:
            stress_indices['moisture_stress'] = np.nan

    except Exception as e:
        print(f"        Error in custom stress indices: {e}")
        stress_indices = {
            'cwsi_proxy': np.nan,
            'cwsi_swir_nir': np.nan,
            'lswi_deficit': np.nan,
            'lswi_ndvi_ratio': np.nan,
            'ndwi_stress': np.nan,
            'moisture_stress': np.nan
        }

    return stress_indices

# ==================== ENHANCED TEMPORAL FEATURES ====================
def calculate_temporal_features(all_phase_data):
    """Calculate temporal features as per desired output format"""
    temporal = {}
    phases = list(all_phase_data.keys())

    try:
        # NDVI Growth Rate (Vegetation - Sowing)
        if 'Sowing' in phases and 'Vegetation' in phases:
            sow = all_phase_data['Sowing']
            veg = all_phase_data['Vegetation']

            sow_ndvi = sow.get('NDVI_mean', np.nan)
            veg_ndvi = veg.get('NDVI_mean', np.nan)

            if not np.isnan(sow_ndvi) and not np.isnan(veg_ndvi):
                temporal['ndvi_growth_rate'] = veg_ndvi - sow_ndvi
            else:
                temporal['ndvi_growth_rate'] = np.nan
        else:
            temporal['ndvi_growth_rate'] = np.nan

        # NDVI Peak (max across all phases)
        ndvi_values = []
        for phase_name, phase_data in all_phase_data.items():
            if 'NDVI_mean' in phase_data and not np.isnan(phase_data['NDVI_mean']):
                ndvi_values.append(phase_data['NDVI_mean'])

        if ndvi_values:
            temporal['ndvi_peak'] = max(ndvi_values)
        else:
            temporal['ndvi_peak'] = np.nan

        # NDVI Decline Rate (Vegetation - Harvesting)
        if 'Vegetation' in phases and 'Harvesting' in phases:
            veg = all_phase_data['Vegetation']
            harv = all_phase_data['Harvesting']

            veg_ndvi = veg.get('NDVI_mean', np.nan)
            harv_ndvi = harv.get('NDVI_mean', np.nan)

            if not np.isnan(veg_ndvi) and not np.isnan(harv_ndvi):
                temporal['ndvi_decline_rate'] = veg_ndvi - harv_ndvi
            else:
                temporal['ndvi_decline_rate'] = np.nan
        else:
            temporal['ndvi_decline_rate'] = np.nan

        # NDVI Season Change (Sowing - Harvesting)
        if 'Sowing' in phases and 'Harvesting' in phases:
            sow = all_phase_data['Sowing']
            harv = all_phase_data['Harvesting']

            sow_ndvi = sow.get('NDVI_mean', np.nan)
            harv_ndvi = harv.get('NDVI_mean', np.nan)

            if not np.isnan(sow_ndvi) and not np.isnan(harv_ndvi):
                temporal['ndvi_season_change'] = sow_ndvi - harv_ndvi
            else:
                temporal['ndvi_season_change'] = np.nan
        else:
            temporal['ndvi_season_change'] = np.nan

    except Exception as e:
        print(f"        Error in temporal features: {e}")
        temporal = {
            'ndvi_growth_rate': np.nan,
            'ndvi_peak': np.nan,
            'ndvi_decline_rate': np.nan,
            'ndvi_season_change': np.nan
        }

    return temporal

# ==================== SINGLE PHASE PROCESSING ====================
def process_single_phase(phase_folder, phase_name, extractor, debug):
    """Process a single phase folder and return features"""
    try:
        # RGB composite + bands
        rgb_image, bands_dict = create_rgb_from_bands(phase_folder, debug=debug)
        if rgb_image is None:
            if debug:
                print(f"          No bands found in {phase_name}")
            return None

        # CNN features
        cnn_features = extractor.extract_features(rgb_image)

        # Spectral indices
        index_stats, raw_indices = extract_spectral_indices(phase_folder, debug=debug)

        # Calculate NDWI from bands if not available
        if 'NDWI' not in raw_indices:
            if 'B03' in bands_dict and 'B08' in bands_dict:
                green_data = bands_dict['B03']
                nir_data = bands_dict['B08']

                valid_mask = (green_data != 0) & (nir_data != 0) & (~np.isnan(green_data)) & (~np.isnan(nir_data))

                if np.sum(valid_mask) > 100:
                    green_valid = green_data[valid_mask]
                    nir_valid = nir_data[valid_mask]

                    ndwi_calculated = (green_valid - nir_valid) / (green_valid + nir_valid + 1e-8)
                    raw_indices['NDWI'] = np.full_like(green_data, np.nan)
                    raw_indices['NDWI'][valid_mask] = ndwi_calculated

        # Calculate LSWI from bands if not available
        if 'LSWI' not in raw_indices:
            if 'B08' in bands_dict and 'B11' in bands_dict:
                nir_data = bands_dict['B08']
                swir_data = bands_dict['B11']

                valid_mask = (nir_data != 0) & (swir_data != 0) & (~np.isnan(nir_data)) & (~np.isnan(swir_data))

                if np.sum(valid_mask) > 100:
                    nir_valid = nir_data[valid_mask]
                    swir_valid = swir_data[valid_mask]

                    lswi_calculated = (nir_valid - swir_valid) / (nir_valid + swir_valid + 1e-8)
                    raw_indices['LSWI'] = np.full_like(nir_data, np.nan)
                    raw_indices['LSWI'][valid_mask] = lswi_calculated

        # Custom stress indices
        stress_indices = calculate_custom_stress_indices(bands_dict, raw_indices)

        # Canopy features
        canopy_features = calculate_canopy_features(bands_dict, raw_indices)

        # Store phase data
        phase_data = {
            'phase': phase_name,
            'cnn_features': cnn_features,
            **index_stats,
            **stress_indices,
            **canopy_features
        }

        if debug:
            print(f"         Extracted: {len(cnn_features)} CNN features, "
                  f"{len(index_stats)} spectral stats, "
                  f"{len(stress_indices)} stress indices, "
                  f"{len(canopy_features)} canopy features")

        return phase_data

    except Exception as e:
        print(f"         Error processing {phase_name}: {e}")
        return None

# ==================== MAIN PROCESSING ====================
def process_all_images(root_dir='Crops', model_name='efficientnet-b4', debug_first=True):
    """Enhanced processing with custom output format"""

    extractor = CNNFeatureExtractor(model_name)
    all_data = []
    first_processed = False

    print("\n" + "="*70)
    print(f" CUSTOM FEATURE EXTRACTION FOR CROP YIELD OPTIMIZATION")
    print(f"   Model: {model_name.upper()}")
    print("="*70 + "\n")

    root_path = Path(root_dir)

    # First pass: collect all phase data per district
    district_phase_data = {}

    for crop_folder in root_path.iterdir():
        if not crop_folder.is_dir():
            continue

        crop_name = crop_folder.name
        print(f"\n Crop: {crop_name}")

        # Check if there are year folders
        subfolders = list(crop_folder.iterdir())
        has_year_folders = any(
            folder.is_dir() and ('-' in folder.name or folder.name.isdigit())
            for folder in subfolders
        )

        if has_year_folders:
            # Structure: Crop/Year/District/Phase/
            for year_folder in crop_folder.iterdir():
                if not year_folder.is_dir():
                    continue
                year_name = year_folder.name
                print(f"   Year: {year_name}")

                for district_folder in year_folder.iterdir():
                    if not district_folder.is_dir():
                        continue
                    district_name = district_folder.name
                    print(f"     District: {district_name}")

                    district_key = f"{crop_name}_{year_name}_{district_name}"
                    district_phase_data[district_key] = {
                        'crop': crop_name,
                        'year': year_name,
                        'district': district_name,
                        'phases': {}
                    }

                    for phase_folder in district_folder.iterdir():
                        if not phase_folder.is_dir():
                            continue
                        phase_name = phase_folder.name
                        print(f"       Phase: {phase_name}")

                        debug = debug_first and not first_processed

                        phase_data = process_single_phase(
                            phase_folder, phase_name, extractor, debug
                        )

                        if phase_data:
                            district_phase_data[district_key]['phases'][phase_name] = phase_data
                            print(f"         Features extracted")

                            if not first_processed:
                                first_processed = True
                        else:
                            print(f"          No features extracted")

        else:
            # Structure: Crop/District/Phase/
            for district_folder in crop_folder.iterdir():
                if not district_folder.is_dir():
                    continue
                district_name = district_folder.name
                print(f"   District: {district_name}")

                district_key = f"{crop_name}_NoYear_{district_name}"
                district_phase_data[district_key] = {
                    'crop': crop_name,
                    'year': None,
                    'district': district_name,
                    'phases': {}
                }

                for phase_folder in district_folder.iterdir():
                    if not phase_folder.is_dir():
                        continue
                    phase_name = phase_folder.name
                    print(f"     Phase: {phase_name}")

                    debug = debug_first and not first_processed

                    phase_data = process_single_phase(
                        phase_folder, phase_name, extractor, debug
                    )

                    if phase_data:
                        district_phase_data[district_key]['phases'][phase_name] = phase_data
                        print(f"       Features extracted")

                        if not first_processed:
                            first_processed = True
                    else:
                        print(f"       No features extracted")

    # Second pass: calculate temporal features and create final dataset
    print(f"\n{'='*70}")
    print(f" CALCULATING TEMPORAL FEATURES...")
    print(f"{'='*70}\n")

    for district_key, data in district_phase_data.items():
        phases = data['phases']
        if not phases:
            continue

        temporal_features = calculate_temporal_features(phases)

        for phase_name, phase_data in phases.items():
            feature_dict = {
                'district_name': data['district'],
                'crop_name': data['crop'],
                'crop_phase': phase_name,
            }

            if data.get('year'):
                feature_dict['year'] = data['year']

            # Add CNN features with proper formatting
            for i, val in enumerate(phase_data['cnn_features']):
                feature_dict[f'CNN_feat_{i+1:04d}'] = float(f"{val:.6f}")

            # Add spectral indices, stress indices, and canopy features
            for key, val in phase_data.items():
                if key not in ['phase', 'cnn_features']:
                    if isinstance(val, (int, float)) and not np.isnan(val):
                        feature_dict[key] = float(f"{val:.6f}")
                    else:
                        feature_dict[key] = val

            # Add temporal features (same for all phases of same district-crop)
            for key, val in temporal_features.items():
                if isinstance(val, (int, float)) and not np.isnan(val):
                    feature_dict[key] = float(f"{val:.6f}")
                else:
                    feature_dict[key] = val

            all_data.append(feature_dict)

    return all_data, extractor.feature_dim

# ==================== ANALYSIS FUNCTION ====================
def analyze_output(df):
    """Analyze the output and provide insights"""
    print(f"\n{'='*70}")
    print(f" OUTPUT ANALYSIS & INSIGHTS")
    print(f"{'='*70}")

    # Basic statistics
    print(f"\n BASIC STATISTICS:")
    print(f"   Total samples: {len(df)}")
    print(f"   Total features: {len(df.columns)}")
    print(f"   Crops: {df['crop_name'].unique().tolist()}")
    print(f"   Districts: {df['district_name'].unique().tolist()}")
    print(f"   Crop phases: {df['crop_phase'].unique().tolist()}")

    # Stress analysis
    print(f"\n  STRESS ANALYSIS:")
    stress_cols = ['cwsi_proxy', 'ndwi_stress', 'moisture_stress', 'canopy_stress']

    for col in stress_cols:
        if col in df.columns:
            mean_stress = df[col].mean()
            max_stress = df[col].max()
            min_stress = df[col].min()

            print(f"   {col}:")
            print(f"      Average: {mean_stress:.3f} | Max: {max_stress:.3f} | Min: {min_stress:.3f}")

            # Interpretation
            if mean_stress < 0.3:
                status = "LOW STRESS "
            elif mean_stress < 0.6:
                status = "MODERATE STRESS "
            else:
                status = "HIGH STRESS "

            print(f"      Status: {status}")

    # Canopy analysis
    print(f"\n CANOPY ANALYSIS:")
    canopy_cols = ['canopy_health', 'canopy_density', 'canopy_water_content', 'canopy_index']

    for col in canopy_cols:
        if col in df.columns:
            mean_val = df[col].mean()
            print(f"   {col}: {mean_val:.3f}")

            # Interpretation based on column type
            if 'health' in col or 'index' in col:
                if mean_val > 0.7:
                    status = "EXCELLENT "
                elif mean_val > 0.5:
                    status = "GOOD "
                elif mean_val > 0.3:
                    status = "FAIR "
                else:
                    status = "POOR "
                print(f"      Status: {status}")

    # Temporal analysis
    print(f"\n TEMPORAL ANALYSIS:")
    temporal_cols = ['ndvi_growth_rate', 'ndvi_decline_rate', 'ndvi_season_change']

    for col in temporal_cols:
        if col in df.columns:
            mean_val = df[col].mean()
            print(f"   {col}: {mean_val:.3f}")

            if 'growth' in col:
                if mean_val > 0.1:
                    status = "STRONG GROWTH "
                elif mean_val > 0:
                    status = "MODERATE GROWTH "
                elif mean_val == 0:
                    status = "NO GROWTH "
                else:
                    status = "NEGATIVE GROWTH "
                print(f"      Status: {status}")

    # Crop-wise analysis
    print(f"\n CROP-WISE ANALYSIS:")
    for crop in df['crop_name'].unique():
        crop_data = df[df['crop_name'] == crop]
        print(f"   {crop}:")
        print(f"      Samples: {len(crop_data)}")

        if 'canopy_index' in df.columns:
            canopy_avg = crop_data['canopy_index'].mean()
            print(f"      Avg Canopy Index: {canopy_avg:.3f}")

        if 'cwsi_proxy' in df.columns:
            stress_avg = crop_data['cwsi_proxy'].mean()
            print(f"      Avg Water Stress: {stress_avg:.3f}")

    # Phase-wise analysis
    print(f"\n PHASE-WISE ANALYSIS:")
    for phase in df['crop_phase'].unique():
        phase_data = df[df['crop_phase'] == phase]
        print(f"   {phase}:")
        print(f"      Samples: {len(phase_data)}")

        if 'NDVI_mean' in df.columns:
            ndvi_avg = phase_data['NDVI_mean'].mean()
            print(f"      Avg NDVI: {ndvi_avg:.3f}")

# ==================== POST-PROCESSING ====================
def post_process_dataframe(df):
    """Post-process dataframe to match desired format"""

    # Ensure all required columns exist
    required_columns = [
        'cwsi_proxy', 'cwsi_swir_nir', 'lswi_deficit', 'lswi_ndvi_ratio',
        'ndwi_stress', 'moisture_stress', 'ndvi_growth_rate', 'ndvi_peak',
        'ndvi_decline_rate', 'ndvi_season_change',
        'canopy_health', 'canopy_density', 'canopy_water_content', 'canopy_stress', 'canopy_index'
    ]

    for col in required_columns:
        if col not in df.columns:
            df[col] = np.nan

    # Fill NaN values with appropriate defaults
    for col in df.columns:
        if df[col].isna().any():
            if 'stress' in col or 'cwsi' in col:
                df[col] = df[col].fillna(0.5)
            elif 'health' in col or 'density' in col or 'index' in col:
                df[col] = df[col].fillna(0.5)
            elif 'water_content' in col:
                df[col] = df[col].fillna(0.5)
            elif 'growth' in col or 'decline' in col or 'change' in col:
                df[col] = df[col].fillna(0.0)
            elif 'ratio' in col:
                df[col] = df[col].fillna(1.0)
            elif 'peak' in col:
                df[col] = df[col].fillna(0.0)
            else:
                df[col] = df[col].fillna(0.0)

    # Reorder columns
    metadata_cols = ['district_name', 'crop_name', 'crop_phase']
    if 'year' in df.columns:
        metadata_cols.append('year')

    cnn_cols = [col for col in df.columns if 'CNN_feat' in col]
    other_cols = [col for col in df.columns if col not in metadata_cols + cnn_cols]

    # Ensure stress and canopy indices are at the end
    stress_cols = [col for col in required_columns if col in df.columns]
    other_cols = [col for col in other_cols if col not in stress_cols] + stress_cols

    df = df[metadata_cols + cnn_cols + other_cols]

    return df

# ==================== MAIN FUNCTION ====================
def main(crops_folder_path=None, model_name='efficientnet-b4', preview_only=True):
    """
    Main function to extract features from Crops folder
    """

    # Find Crops folder
    if crops_folder_path is None:
        if mount_google_drive():
            crops_folder_path = find_crops_folder()
            if crops_folder_path is None:
                print(" Please provide the path to Crops folder manually")
                return None
        else:
            print(" Could not mount Google Drive")
            return None
    else:
        if not os.path.exists(crops_folder_path):
            print(f" Folder not found: {crops_folder_path}")
            return None

    print(f"\n Using folder: {crops_folder_path}")

    # Extract features
    features_list, feature_dim = process_all_images(crops_folder_path, model_name, debug_first=True)

    if not features_list:
        print("\n No features extracted!")
        return None

    df = pd.DataFrame(features_list)
    df = post_process_dataframe(df)

    print("\n" + "="*70)
    print(f" FINAL FEATURE SUMMARY:")
    print("="*70)
    print(f" Total samples: {len(df)}")
    print(f" Total features: {len(df.columns)}")
    print("="*70)

    print("\n OUTPUT PREVIEW (First 5 rows):")
    print(df.head())

    # Perform analysis
    analyze_output(df)

    if preview_only:
        print("\n  Preview mode - CSV not saved")
        return df

    # Save CSV
    output_file = f'crop_features_custom_format.csv'
    df.to_csv(output_file, index=False)
    print(f"\n CSV saved: {output_file}")

    # Save to Drive if possible
    if '/content/drive/' in crops_folder_path:
        drive_folder = os.path.dirname(crops_folder_path)
        drive_output = os.path.join(drive_folder, output_file)
        df.to_csv(drive_output, index=False)
        print(f" Drive copy: {drive_output}")

    return df

# ==================== RUN ====================
if __name__ == "__main__":

    print("🔍 STARTING FEATURE EXTRACTION WITH CANOPY ANALYSIS\n")

    # Auto-search for Crops folder in Drive
    df = main(
        crops_folder_path=None,
        model_name='efficientnet-b4',
        preview_only=False
    )

    if df is not None:
        print(f"\n FEATURE EXTRACTION & ANALYSIS COMPLETED SUCCESSFULLY!")
        print(f" Final dataset shape: {df.shape}")
        print(f" Samples: {len(df)}")
        print(f" Features: {len(df.columns)}")
        print(f" Canopy features included: ")
        print(f" Analysis provided: ")
    else:
        print(f"\n FEATURE EXTRACTION FAILED!")

TensorFlow: 2.19.0 | GPU: 0
🔍 STARTING FEATURE EXTRACTION WITH CANOPY ANALYSIS

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Drive mounted!
🔍 Searching for Crops folder in Google Drive...
 Found: /content/drive/MyDrive/Final Year Project/Crops

 Using folder: /content/drive/MyDrive/Final Year Project/Crops

 Loading EFFICIENTNET-B4...
71686520/71686520 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
 Loaded! Features: 1792

 CUSTOM FEATURE EXTRACTION FOR CROP YIELD OPTIMIZATION
   Model: EFFICIENTNET-B4


 Crop: Tomato
   District: Quetta
     Phase: Harvesting
      ✓ B02: 2023-02-01-00_00_2023-03-10-23_59_Sentinel-2_L2A_B02_(Raw).tiff
      ✓ B03: 2023-02-01-00_00_2023-03-10-23_59_Sentinel-2_L2A_B03_(Raw).tiff
      ✓ B04: 2023-02-01-00_00_2023-03-10-23_59_Sentinel-2_L2A_B04_(Raw).tiff
      ✓ B08: 2023-02-01-00_00_2023-03-10-23_59_Sentinel-2_L2A_B8A_(Raw).tiff
      ✓ B11: 2023-02-01-00_00_2023-03-10-23_59_Sentine

       Features extracted
     Phase: Sowing


       Features extracted
     Phase: Harvesting


       Features extracted
   District: Hyderabad
     Phase: Vegetation
       Features extracted
     Phase: Harvesting
       Features extracted
     Phase: Sowing
       Features extracted
   District: Mirpurkhas
     Phase: Vegetation
       Features extracted
     Phase: Sowing
       Features extracted
     Phase: Harvesting
       Features extracted
   District: D.G. Khan
     Phase: Sowing
       Features extracted
     Phase: Vegetation
       Features extracted
     Phase: Harvesting
       Features extracted
   District: Bahawalpur
     Phase: Sowing
       Features extracted
     Phase: Vegetation
       Features extracted
     Phase: Harvesting
       Features extracted
   District: Khanewal
     Phase: Vegetation
       Features extracted
     Phase: Sowing
       Features extracted
     Phase: Harvesting
       Features extracted
   District: Thatta
     Phase: Sowing
       Features extracted
     Phase: Vegetation
       Features extracted
     Phase: Harvesting
      

       Features extracted
   District: Jhang
     Phase: Sowing
       Features extracted
     Phase: Harvest
       Features extracted
     Phase: Vegetation
       Features extracted
   District: Nankana Sahib
     Phase: Sowing


...

       Features extracted
     Phase: Harvesting


       Features extracted
   District: Hyderabad
     Phase: Vegetation
       Features extracted
     Phase: Sowing
       Features extracted
     Phase: Harvesting
       Features extracted
   District: Mirpurkhas
     Phase: Vegetation
       Features extracted
     Phase: Sowing
       Features extracted
     Phase: Harvesting
       Features extracted
   District: Thatta
     Phase: Sowing
       Features extracted
     Phase: Harvesting
       Features extracted
     Phase: Vegetation
       Features extracted
   District: Khanewal
     Phase: Harvesting
       Features extracted
     Phase: Sowing
       Features extracted
     Phase: Vegetation
       Features extracted
   District: D.G. Khan
     Phase: Vegetation
       Features extracted
     Phase: Sowing
       Features extracted
     Phase: Harvesting
       Features extracted

 Crop: Potato
   District: Quetta
     Phase: Sowing
       Features extracted
     Phase: Vegitative
       Features extracted
     Phase: Harves

       Features extracted
     Phase: Vegitative


       Features extracted
     Phase: Sowing


       Features extracted
   District: Rahimyar Khan
     Phase: Vegetation
       Features extracted
     Phase: Harvesting
       Features extracted
     Phase: Sowing
       Features extracted
   District: Turbat
     Phase: Vegetation
       Features extracted
     Phase: Harvesting
       Features extracted
     Phase: Sowing
       Features extracted

 Crop: Cotton
   District: Ghotki
     Phase: Sowing
       Features extracted
     Phase: Vegitative
       Features extracted
     Phase: Harvesting
       Features extracted
   District: Sukkur
     Phase: Vegitative
       Features extracted
     Phase: Harvesting
       Features extracted
     Phase: Sowing
       Features extracted
   District: Khairpur
     Phase: Harvesting
       Features extracted
     Phase: Sowing
       Features extracted
     Phase: Vegitative
       Features extracted
   District: Rahimyar Khan
     Phase: Sowing
       Features extracted
     Phase: Vegetation
       Features extracted
     Phase: Ha

In [ ]:
df = pd.read_csv('crop_features_custom_format.csv')

In [ ]:
df.shape

(197, 1852)